In [ ]:
# 3_model_training

import time
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, GBTClassifier, LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print(">>> Mounting Google Drive...")
drive.mount('/content/drive')

spark = SparkSession.builder \
    .appName("HIGGS_ModelTraining") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory", "6g") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir("/tmp/checkpoints")

# 1. Load data
features_input_path = "/content/drive/MyDrive/Bigdata/HIGGS_features.parquet"
df_scaled = spark.read.parquet(features_input_path)

# 2. Memory Management & Splitting
model_data = df_scaled.cache()
print(f"  Data ready for ML Training: {model_data.count()} rows")

train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)
print("   Data successfully split into Train (80%) and Test (20%).")

print("\n>>> STEP 4: Training 4 Big Data Models")

# 3. Define Algorithms & Evaluators
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)
svm = LinearSVC(labelCol="label", featuresCol="features", maxIter=10)
gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=10, seed=42)

eval_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
eval_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

# 4. Hyperparameter Tuning
paramGrid_rf = ParamGridBuilder().addGrid(rf.numTrees, [10, 20]).build()
cv_rf = CrossValidator(estimator=rf, estimatorParamMaps=paramGrid_rf, evaluator=eval_auc, numFolds=2, parallelism=2)

models = [
    ("Random Forest (CV)", cv_rf),
    ("Gradient Boosted", gbt),
    ("Logistic Reg", lr),
    ("Linear SVM", svm)
]

print(f"\n{'Algorithm':<20} | {'AUC':<7} | {'Acc':<7} | {'Time'}")
print("-" * 50)

best_higgs_model_vraj = None
best_predictions = None

# 5. Training Loop
for name, algo in models:
    try:
        start_time = time.time()
        model = algo.fit(train_data)
        preds = model.transform(test_data)

        auc = eval_auc.evaluate(preds)
        acc = eval_acc.evaluate(preds)

        print(f"{name:<20} | {auc:.4f} | {acc:.4f} | {(time.time() - start_time):.1f}s")

        if name == "Gradient Boosted":
            best_higgs_model_vraj = model
            best_predictions = preds

    except Exception as e:
        print(f"{name:<20} | FAILED: {str(e)[:50]}")

# 6. SERIALIZATION & HANDOFF
if best_higgs_model_vraj:
    model_path = "/content/drive/MyDrive/Bigdata/higgs_gbt_model"
    best_higgs_model_vraj.write().overwrite().save(model_path)

    # Save predictions
    preds_output_path = "/content/drive/MyDrive/Bigdata/HIGGS_best_predictions.parquet"
    best_predictions.write.mode("overwrite").parquet(preds_output_path)
    print(f"\nSUCCESS: GBT Model and Predictions serialized")